# Ingeniería de características (Feature Engineering)

**Proyecto:** House Pricing — House Prices: Advanced Regression Techniques (Kaggle)

**Notebook:** `2.0-ae-ingenieria-de-caracteristicas.ipynb` (convención Cookiecutter Data Science)

Este notebook toma las 30 features seleccionadas en el EDA (`MODEL_FEATURES` en `pipeline_config.yaml`),
construye un **pipeline de sklearn** con transformadores de **feature-engine** y genera los datasets
procesados que serán el input del notebook de modelos.

**Decisiones acordadas:**

| Paso | Decisión |
|---|---|
| Faltantes | >20% en train → se descarta la variable (aplica a numéricas y categóricas) |
| 1. Imputación categóricas | Moda (`CategoricalImputer`) |
| 1b. Imputación numéricas | ≤5% → mediana (`MeanMedianImputer`); 5–20% → muestreo aleatorio de la distribución real (`RandomSampleImputer`, semilla por observación) |
| 2. Codificación | Ordinal manual para calidad (Po=0 … Ex=4); one-hot para nominales |
| 3. Outliers | Eliminar las 2 casas con `GrLivArea > 4000` y precio bajo + `Winsorizer` IQR (1.5, cola derecha) en las continuas de área |
| 4. Transformación | `YeoJohnsonTransformer` en las 6 continuas con skew > 0.75 en `train.csv` |
| 5. Escalado | `StandardScaler` en numéricas; las dummies one-hot quedan en 0/1 |
| Target | `SalePrice_log = log1p(SalePrice)` |
| Split | train/val 80/20 (`random_state=42`); el pipeline se ajusta solo con train |
| Salida | CSV en `data/processed/` y pipeline ajustado en `models/` (joblib) |

## 0. Librerías y configuración

Importa las librerías, agrega la raíz del proyecto al path para usar `house_pricing.config` y carga
la lista de features desde `pipeline_config.yaml`.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from feature_engine.encoding import OneHotEncoder
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer, RandomSampleImputer
from feature_engine.outliers import Winsorizer
from feature_engine.selection import DropFeatures
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper

sys.path.append(str(Path.cwd().parent))

from house_pricing.config import MODELS_DIR, PROCESSED_DATA_DIR, PROJ_ROOT, RAW_DATA_DIR

RANDOM_STATE = 42

with open(PROJ_ROOT / "pipeline_config.yaml", encoding="utf-8") as f:
    MODEL_FEATURES = yaml.safe_load(f)["MODEL_FEATURES"]

pd.set_option("display.max_columns", 100)
print(f"Features en pipeline_config.yaml: {len(MODEL_FEATURES)}")

2026-11-09 21:14:15.735 | INFO     | house_pricing.config:<module>:11 - PROJ_ROOT path is: C:\Users\Anderson Elias\OneDrive - Universidad Galileo\8vo. Trimestre\Product Development - B\house_pricing


Features en pipeline_config.yaml: 30


## 1. Carga de datos y eliminación de outliers anómalos

Carga `train.csv` y `test.csv`. Antes de cualquier otra cosa elimina del train las **2 casas con
`GrLivArea > 4000` y precio bajo**: son los outliers conocidos de este dataset (casas enormes vendidas
muy por debajo de la tendencia). Las otras 2 casas con más de 4000 pies² son caras y siguen la tendencia,
por eso se conservan. Esto se hace fuera del pipeline porque un pipeline de sklearn no puede eliminar filas.

In [2]:
train_raw = pd.read_csv(RAW_DATA_DIR / "train.csv")
test_raw = pd.read_csv(RAW_DATA_DIR / "test.csv")

anomalous = (train_raw["GrLivArea"] > 4000) & (train_raw["SalePrice"] < 300000)
display(train_raw.loc[anomalous, ["Id", "GrLivArea", "SalePrice"]])

train_raw = train_raw.loc[~anomalous].reset_index(drop=True)
print(f"train: {train_raw.shape} (se eliminaron {anomalous.sum()} filas) | test: {test_raw.shape}")

,Id,GrLivArea,SalePrice
523,524,4676,184750
1298,1299,5642,160000


train: (1458, 81) (se eliminaron 2 filas) | test: (1459, 80)


## 2. Separación train / validación y target

Separa el train en 80% entrenamiento y 20% validación. El pipeline se ajustará **solo** con el 80%
para que las estadísticas (medianas, modas, límites de outliers, parámetros de Yeo-Johnson, media y
desviación del escalado) no vean los datos de validación. El target se guarda como
`SalePrice_log = log1p(SalePrice)` para reducir su sesgo (skew 1.88); en el notebook de modelos se
revierte con `np.expm1`.

In [3]:
X = train_raw[MODEL_FEATURES]
y = np.log1p(train_raw["SalePrice"]).rename("SalePrice_log")

X_train, X_val, y_train, y_val, id_train, id_val = train_test_split(
    X, y, train_raw["Id"], test_size=0.2, random_state=RANDOM_STATE
)
X_test, id_test = test_raw[MODEL_FEATURES], test_raw["Id"]

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Skew SalePrice: {train_raw['SalePrice'].skew():.2f} -> SalePrice_log: {y.skew():.2f}")

X_train: (1166, 30) | X_val: (292, 30) | X_test: (1459, 30)
Skew SalePrice: 1.88 -> SalePrice_log: 0.12


## 3. Clasificación de variables y regla de faltantes

Separa las features en categóricas, discretas (≤ 25 valores únicos) y continuas con el mismo criterio
del EDA, y aplica la regla de faltantes acordada sobre el **train**:

- **> 20%** de faltantes → la variable se descarta.
- **5% – 20%** → imputación por muestreo aleatorio (solo numéricas).
- **≤ 5%** → imputación por mediana (numéricas).
- **Categóricas** con ≤ 20% → imputación por moda, sin importar su porcentaje.

Las variables sin faltantes en train también se incluyen en la imputación, porque `test.csv` sí tiene
algunos NaN en ellas (por ejemplo `GarageCars` o `KitchenQual`).

In [4]:
DISCRETE_THRESHOLD = 25

categorical = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
numeric = X_train.select_dtypes(include=[np.number]).columns.tolist()
nunique = X_train[numeric].nunique()
discrete = [c for c in numeric if nunique[c] <= DISCRETE_THRESHOLD]
continuous = [c for c in numeric if nunique[c] > DISCRETE_THRESHOLD]

missing_pct = X_train.isna().mean() * 100
missing_summary = pd.DataFrame({
    "missing_pct": missing_pct.round(2),
    "tipo": ["categórica" if c in categorical else "discreta" if c in discrete else "continua" for c in X_train.columns],
})
missing_summary["tratamiento"] = np.select(
    [missing_summary["missing_pct"] > 20, missing_summary["tipo"] == "categórica", missing_summary["missing_pct"] > 5],
    ["descartar", "moda", "muestreo aleatorio"],
    default="mediana",
)

drop_vars = missing_summary.index[missing_summary["tratamiento"] == "descartar"].tolist()
random_vars = missing_summary.index[missing_summary["tratamiento"] == "muestreo aleatorio"].tolist()
cat_vars = [c for c in categorical if c not in drop_vars + random_vars]
median_vars = [c for c in numeric if c not in drop_vars + random_vars]

print(f"Descartar: {drop_vars}")
print(f"Muestreo aleatorio: {random_vars}")
missing_summary.query("missing_pct > 0").sort_values("missing_pct", ascending=False)

Descartar: ['FireplaceQu']


Muestreo aleatorio: ['GarageYrBlt', 'LotFrontage']


,missing_pct,tipo,tratamiento
FireplaceQu,47.34,categórica,descartar
LotFrontage,18.27,continua,muestreo aleatorio
GarageFinish,5.83,categórica,moda
GarageYrBlt,5.83,continua,muestreo aleatorio
GarageType,5.83,categórica,moda
BsmtQual,2.74,categórica,moda
BsmtFinType1,2.74,categórica,moda
MasVnrArea,0.51,continua,mediana


## 4. Paso 1 — Imputación de variables categóricas

Elimina las variables con más de 20% de faltantes (`DropFeatures`) e imputa los NaN de las categóricas
restantes con la **moda** de cada variable en train (`CategoricalImputer`).

In [5]:
drop_high_missing = DropFeatures(features_to_drop=drop_vars)
categorical = [c for c in categorical if c not in drop_vars]

categorical_imputer = CategoricalImputer(imputation_method="frequent", variables=cat_vars)
categorical_imputer

,imputation_method,'frequent'
,variables,"['Neighborhood', 'ExterQual', ...]"
,fill_value,'Missing'
,return_object,False
,ignore_format,False


## 5. Paso 1b — Imputación de variables numéricas

Imputa las numéricas según el porcentaje de faltantes en train:

- **Muestreo aleatorio** (`RandomSampleImputer`): rellena cada NaN con un valor tomado al azar de los
  valores observados de la variable, así la distribución imputada se aproxima a la real.
- **Mediana** (`MeanMedianImputer`): para las variables con ≤ 5% de faltantes.

El muestreo usa una **semilla por observación** (`seed="observation"`): la semilla de cada fila se
calcula sumando sus valores de `YearBuilt`, `OverallQual` y `GrLivArea` (columnas sin faltantes). Así
una misma casa recibe siempre el mismo valor imputado, sin importar qué otras casas se procesen junto
a ella. Con una semilla global, el valor dependía de la posición de la fila dentro del lote, y el
precio predicho de una casa cambiaba según la petición en la que viniera.

In [6]:
SEED_VARS = ["YearBuilt", "OverallQual", "GrLivArea"]

random_sample_imputer = RandomSampleImputer(
    variables=random_vars, random_state=SEED_VARS, seed="observation", seeding_method="add"
)
median_imputer = MeanMedianImputer(imputation_method="median", variables=median_vars)

print(f"RandomSampleImputer: {random_vars} (semilla por observación: {SEED_VARS})")
print(f"MeanMedianImputer:   {median_vars}")

RandomSampleImputer: ['GarageYrBlt', 'LotFrontage'] (semilla por observación: ['YearBuilt', 'OverallQual', 'GrLivArea'])
MeanMedianImputer:   ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'Fireplaces', 'HalfBath', 'BsmtFullBath', 'BedroomAbvGr', 'KitchenAbvGr', 'PoolArea']


## 6. Paso 2 — Codificación de variables categóricas

- **Ordinales de calidad** (`ExterQual`, `BsmtQual`, `KitchenQual`, `HeatingQC`): se mapean según su
  orden real `Po=0 < Fa=1 < TA=2 < Gd=3 < Ex=4`. Se usa el `OrdinalEncoder` de sklearn con las
  categorías explícitas, envuelto en `SklearnTransformerWrapper` de feature-engine para conservar el
  DataFrame. Como después se estandarizan, empezar en 0 o en 1 da exactamente el mismo resultado.
- **Nominales** (`Neighborhood`, `Foundation`, `GarageType`, `GarageFinish`, `BsmtFinType1`): one-hot
  (`OneHotEncoder` de feature-engine), una columna 0/1 por categoría vista en train.

In [7]:
QUALITY_ORDER = ["Po", "Fa", "TA", "Gd", "Ex"]
quality_vars = [c for c in ["ExterQual", "BsmtQual", "KitchenQual", "FireplaceQu", "HeatingQC"] if c in categorical]
nominal_vars = [c for c in categorical if c not in quality_vars]

ordinal_encoder = SklearnTransformerWrapper(
    transformer=OrdinalEncoder(categories=[QUALITY_ORDER] * len(quality_vars)),
    variables=quality_vars,
)
one_hot_encoder = OneHotEncoder(variables=nominal_vars)

print(f"Ordinal ({QUALITY_ORDER}): {quality_vars}")
print(f"One-hot: {nominal_vars}")

Ordinal (['Po', 'Fa', 'TA', 'Gd', 'Ex']): ['ExterQual', 'BsmtQual', 'KitchenQual', 'HeatingQC']
One-hot: ['Neighborhood', 'GarageFinish', 'Foundation', 'GarageType', 'BsmtFinType1']


## 7. Paso 3 — Tratamiento de outliers

Las 2 casas anómalas ya se eliminaron en la sección 1. Aquí se aplica **capping** con `Winsorizer`:
los valores por encima de `Q3 + 1.5 × IQR` (calculado en train) se reemplazan por ese límite.
Solo se recorta la **cola derecha** y solo en las continuas de área/medida, para no alterar los ceros
(casas sin sótano, sin revestimiento, etc.) ni los años de construcción.

In [8]:
outlier_vars = [c for c in continuous if c not in ["YearBuilt", "YearRemodAdd", "GarageYrBlt"]]

winsorizer = Winsorizer(capping_method="iqr", tail="right", fold=1.5, variables=outlier_vars)
print(f"Winsorizer (IQR 1.5, cola derecha): {outlier_vars}")

Winsorizer (IQR 1.5, cola derecha): ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']


## 8. Paso 4 — Transformación de variables continuas

Aplica `YeoJohnsonTransformer` a las 6 continuas que tenían **skew > 0.75** en el `train.csv` completo
(`GrLivArea`, `TotalBsmtSF`, `1stFlrSF`, `MasVnrArea`, `BsmtFinSF1`, `LotFrontage`). La tabla muestra el
skew en el split de train: tras eliminar las 2 casas anómalas, `TotalBsmtSF` baja a ~0.45, pero se
mantiene en la lista acordada. Yeo-Johnson busca para
cada variable el exponente que la acerca más a una distribución normal y, a diferencia de Box-Cox o del
logaritmo, acepta ceros (`MasVnrArea`, `BsmtFinSF1` y `TotalBsmtSF` los tienen).

In [9]:
skewed_vars = ["GrLivArea", "TotalBsmtSF", "1stFlrSF", "MasVnrArea", "BsmtFinSF1", "LotFrontage"]

skew_train = X_train[continuous].skew().sort_values(ascending=False)

yeo_johnson = YeoJohnsonTransformer(variables=skewed_vars)
print(f"YeoJohnsonTransformer: {skewed_vars}")
skew_train.round(2).to_frame("skew_train")

YeoJohnsonTransformer: ['GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']


,skew_train
MasVnrArea,2.44
LotFrontage,1.76
GrLivArea,1.08
1stFlrSF,0.89
BsmtFinSF1,0.76
TotalBsmtSF,0.45
GarageArea,0.16
YearRemodAdd,-0.50
YearBuilt,-0.61
GarageYrBlt,-0.64


## 9. Paso 5 — Escalado de variables

Estandariza con `StandardScaler` (media 0, desviación 1) las variables numéricas: continuas, discretas
y las ordinales de calidad ya codificadas. Las dummies one-hot se quedan en 0/1 porque escalarlas no
aporta y les quita su interpretación.

In [10]:
scale_vars = continuous + discrete + quality_vars

scaler = SklearnTransformerWrapper(transformer=StandardScaler(), variables=scale_vars)
print(f"StandardScaler ({len(scale_vars)} variables): {scale_vars}")

StandardScaler (24 variables): ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage', 'OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'Fireplaces', 'HalfBath', 'BsmtFullBath', 'BedroomAbvGr', 'KitchenAbvGr', 'PoolArea', 'ExterQual', 'BsmtQual', 'KitchenQual', 'HeatingQC']


## 10. Construcción y ajuste del pipeline

Une todos los pasos en un `Pipeline` de sklearn, en este orden: descartar → imputar categóricas →
imputar numéricas → codificar → capping de outliers → Yeo-Johnson → escalar. Se ajusta **solo con
X_train** y luego transforma train, validación y test con los mismos parámetros.

In [11]:
fe_pipeline = Pipeline(steps=[
    ("drop_high_missing", drop_high_missing),
    ("categorical_imputer", categorical_imputer),
    ("random_sample_imputer", random_sample_imputer),
    ("median_imputer", median_imputer),
    ("ordinal_encoder", ordinal_encoder),
    ("one_hot_encoder", one_hot_encoder),
    ("winsorizer", winsorizer),
    ("yeo_johnson", yeo_johnson),
    ("scaler", scaler),
])

fe_pipeline.fit(X_train)

X_train_fe = fe_pipeline.transform(X_train)
X_val_fe = fe_pipeline.transform(X_val)
X_test_fe = fe_pipeline.transform(X_test)

for name, df in [("train", X_train_fe), ("val", X_val_fe), ("test", X_test_fe)]:
    print(f"{name:5s} shape: {df.shape} | NaN: {df.isna().sum().sum()}")

fe_pipeline

train shape: (1166, 70) | NaN: 0
val   shape: (292, 70) | NaN: 0
test  shape: (1459, 70) | NaN: 0


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('drop_high_missing', ...), ('categorical_imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,features_to_drop,['FireplaceQu']
Name,Type,Value
feature_names_in_,list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
features_to_drop_,list,['Fi...Qu']


## 11. Validación de las transformaciones

Compara en train el skew de las continuas antes y después del pipeline, y verifica que las numéricas
escaladas tengan media ≈ 0 y desviación ≈ 1.

In [12]:
skew_check = pd.DataFrame({
    "skew_original": X_train[continuous].skew(),
    "skew_transformado": X_train_fe[continuous].skew(),
}).round(2).sort_values("skew_original", ascending=False)
display(skew_check)

X_train_fe[scale_vars].agg(["mean", "std"]).T.round(3).head(10)

,skew_original,skew_transformado
MasVnrArea,2.44,0.43
LotFrontage,1.76,-0.01
GrLivArea,1.08,-0.01
1stFlrSF,0.89,-0.00
BsmtFinSF1,0.76,-0.44
TotalBsmtSF,0.45,-0.19
GarageArea,0.16,-0.06
YearRemodAdd,-0.50,-0.50
YearBuilt,-0.61,-0.61
GarageYrBlt,-0.64,-0.64


,mean,std
GrLivArea,-0.0,1.0
GarageArea,0.0,1.0
TotalBsmtSF,-0.0,1.0
1stFlrSF,0.0,1.0
YearBuilt,-0.0,1.0
YearRemodAdd,-0.0,1.0
GarageYrBlt,0.0,1.0
MasVnrArea,0.0,1.0
BsmtFinSF1,0.0,1.0
LotFrontage,-0.0,1.0


## 12. Guardado de datasets y pipeline

Guarda en `data/processed/` los tres datasets transformados, con `Id` como primera columna y
`SalePrice_log` como última en train y val (test no tiene target). El pipeline ajustado se guarda en
`models/fe_pipeline.joblib` para aplicarlo a datos nuevos sin volver a ajustarlo.

In [13]:
train_fe = pd.concat([id_train, X_train_fe, y_train], axis=1)
val_fe = pd.concat([id_val, X_val_fe, y_val], axis=1)
test_fe = pd.concat([id_test, X_test_fe], axis=1)

outputs = {"train_fe.csv": train_fe, "val_fe.csv": val_fe, "test_fe.csv": test_fe}
for filename, df in outputs.items():
    df.to_csv(PROCESSED_DATA_DIR / filename, index=False)
    print(f"{filename}: {df.shape}")

joblib.dump(fe_pipeline, MODELS_DIR / "fe_pipeline.joblib")
print(f"Pipeline guardado en {MODELS_DIR / 'fe_pipeline.joblib'}")

train_fe.head()

train_fe.csv: (1166, 72)
val_fe.csv: (292, 72)
test_fe.csv: (1459, 71)
Pipeline guardado en C:\Users\Anderson Elias\OneDrive - Universidad Galileo\8vo. Trimestre\Product Development - B\house_pricing\models\fe_pipeline.joblib


,Id,GrLivArea,GarageArea,TotalBsmtSF,1stFlrSF,YearBuilt,YearRemodAdd,GarageYrBlt,MasVnrArea,BsmtFinSF1,LotFrontage,OverallQual,GarageCars,FullBath,TotRmsAbvGrd,Fireplaces,HalfBath,BsmtFullBath,BedroomAbvGr,KitchenAbvGr,PoolArea,ExterQual,BsmtQual,KitchenQual,HeatingQC,Neighborhood_NAmes,Neighborhood_Gilbert,Neighborhood_SWISU,Neighborhood_CollgCr,Neighborhood_IDOTRR,Neighborhood_NWAmes,Neighborhood_Somerst,Neighborhood_Sawyer,Neighborhood_Crawfor,Neighborhood_Timber,Neighborhood_SawyerW,Neighborhood_Edwards,Neighborhood_NridgHt,Neighborhood_Veenker,Neighborhood_BrkSide,Neighborhood_Mitchel,Neighborhood_OldTown,Neighborhood_NoRidge,Neighborhood_ClearCr,Neighborhood_BrDale,Neighborhood_Blmngtn,Neighborhood_NPkVill,Neighborhood_MeadowV,Neighborhood_StoneBr,Neighborhood_Blueste,GarageFinish_RFn,GarageFinish_Unf,GarageFinish_Fin,Foundation_CBlock,Foundation_PConc,Foundation_BrkTil,Foundation_Slab,Foundation_Wood,Foundation_Stone,GarageType_Attchd,GarageType_Detchd,GarageType_BuiltIn,GarageType_Basment,GarageType_CarPort,GarageType_2Types,BsmtFinType1_Rec,BsmtFinType1_Unf,BsmtFinType1_BLQ,BsmtFinType1_ALQ,BsmtFinType1_LwQ,BsmtFinType1_GLQ,SalePrice_log
254,255,-0.290454,-0.837013,0.680123,0.564625,-0.459933,-1.346989,-0.863730,-0.817138,0.970944,0.043868,-0.781412,-1.017111,-1.014654,-0.950349,-0.951228,-0.761581,1.098561,0.148858,-0.226848,-0.064936,-0.665758,-0.833003,-0.755676,-1.165102,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,11.884496
1065,1067,0.272079,-0.423521,-0.586026,-1.057384,0.731291,0.447637,0.598674,-0.817138,-1.369688,-0.519068,-0.053675,0.318420,0.783350,0.294715,0.600005,1.216154,-0.813263,0.148858,-0.226848,-0.064936,1.078678,0.656692,-0.755676,-0.128072,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,12.089544
864,866,-1.112249,2.086279,-0.076419,-0.323306,-0.029769,-0.716444,-0.213772,1.048315,0.896636,1.081989,-0.781412,0.318420,-1.014654,-0.950349,-0.951228,-0.761581,1.098561,0.148858,-0.226848,-0.064936,-0.665758,-0.833003,-0.755676,-1.165102,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,11.908347
798,800,0.653543,-1.096647,-0.760581,-0.392274,-1.121725,-1.686513,-1.594932,1.293775,0.652100,-0.468160,-0.781412,-1.017111,-1.014654,0.294715,2.151238,1.216154,1.098561,0.148858,-0.226848,-0.064936,-0.665758,0.656692,0.766117,0.908958,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,12.072547
380,381,0.508863,-0.769700,-0.017171,-0.246164,-1.551889,-1.686513,-2.204267,-0.817138,0.119562,-0.974565,-0.781412,-1.017111,0.783350,-0.327817,0.600005,-0.761581,-0.813263,0.148858,-0.226848,-0.064936,-0.665758,-0.833003,0.766117,-1.165102,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,11.751950
